In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


In [ ]:
# 3. Create DataLoaders


from torch.utils.data import DataLoader

# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
# DataLoader for test/validation data
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")


In [ ]:
# 5. Display sample images


In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer
        self.layer4 = nn.Linear(hidden_dim, 1)

        # ReLU activation
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        a1 = self.relu(self.layer1(x))

        # Second hidden layer
        a2 = self.relu(self.layer2(a1))

        # Third hidden layer
        a3 = self.relu(self.layer3(a2))

        # Output layer
        output = self.layer4(a3)

        return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device).view(-1, 1).to(device)        # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for X_batch, y_batch in test_loader:
            # Move batch to the selected device
            X_batch = X_batch.to(device)
            y_batch = y_batch.view(-1, 1).to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Binary predictions
            predicted = (outputs > 0.5).float()

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy


In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 14                # Design choice

# Instantiate model
model = NN4Layer(input_dim, hidden_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

learning_rate = 0.001
# Define criterion (loss function) - using CrossEntropyLoss as model now outputs logits
criterion = nn.MSELoss()
# Define optimizer
optimizer = AdamW(model.parameters(), lr = learning_rate)


In [ ]:
# Task 5: Start training for 20 epochs:
# Device configuration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

num_epochs = 20

# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: